1. 문서의 내용을 읽는다.
2. 문서를 쪼갠다.
    - 토큰 수 초과로 답변을 생성하지 못할 수 있고
    - 문서가 길면 (인풋이 길면) 답변 생성이 오래 걸린다.
3. 임베딩(벡터로 변환) -> 벡터 데이터베이스에 저장
4. 질문이 있을 때, 벡터 데이터베이스에서 유사도 검색
5. 유사도 검색으로 가져온 문서를 LLM에 질문과 같이 전달

In [ ]:
# .docx (Word 문서) 파일에서 텍스트를 추출하는 라이브러리 설치
%pip install -qU docx2txt langchain_community

# 텍스트 분할을 위한 라이브러리 설치
%pip install -qU langchain-text-splitters

In [ ]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
  # chunk_size = 청크 크기 (최대 문자 수)
  chunk_size=1000,
  # chunk_overlap = 인접한 청크끼리 겹치는 정도 (이전 내용 일부를 다음 청크에 포함)
  chunk_overlap=200
)

# .docx 파일에서 텍스트를 추출하여 문서 객체로 로드
loader = Docx2txtLoader("./tax_docs/tax_with_markdown.docx")
# 텍스트를 추출한 후, 텍스트 분할기를 사용하여 문서를 청크로 나누어 리스트 형태로 반환
document_list = loader.load_and_split(text_splitter=text_splitter)

In [ ]:
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings

# OepnAI를 임베딩 하려면 환경변수에 OPENAI_API_KEY가 설정되어 있어야 한다
load_dotenv()

embedding = OpenAIEmbeddings(model="text-embedding-3-large")

In [ ]:
# Pinecone 설치
%pip install --upgrade --quiet \
    langchain-pinecone \
    langchain-openai  \
    langchain \
    pinecone-notebooks

In [ ]:
import os
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore

index_name = 'tax-index'
pinecone_api_key = os.environ.get("PINECONE_API_KEY")

pc = Pinecone(api_key=pinecone_api_key)

database = PineconeVectorStore.from_documents(document_list, embedding, index_name=index_name)

In [ ]:
query = '연봉 4천만원인 직장인의 소득세는 얼마인가요?'

retrieved_docs = database.similarity_search(query, k=3)

In [ ]:
from langchain_openai import ChatOpenAI

# LLM 모델 초기화
llm = ChatOpenAI(model="gpt-5.4")

In [ ]:
# RetrieverQA Chain을 사용하여 효과적으로 검색하기
%pip install -U langchain langchainhub --quiet

In [ ]:
from dotenv import load_dotenv
from langsmith import Client

import os

load_dotenv()

LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY")
client = Client(api_key=LANGSMITH_API_KEY)
prompt = client.pull_prompt("rlm/rag-prompt", include_model=True)

In [ ]:
from langchain_classic.chains import RetrievalQA

# RetrievalQA: 질문이 들어오면 (1)벡터DB에서 관련 문서 검색 → (2)프롬프트에 문서+질문 조합 → (3)LLM으로 답변 생성을 자동으로 처리하는 체인
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    # as_retriever()는 벡터DB를 통일된 Retriever 인터페이스로 변환 (Chroma, FAISS 등 DB 교체 시 코드 변경 최소화)
    retriever=database.as_retriever(),
    # LangSmith에서 가져온 RAG 프롬프트 템플릿을 체인에 적용
    chain_type_kwargs={"prompt": prompt}
)

In [ ]:
ai_message = qa_chain({"query": query})

In [ ]:
ai_message